## JOIN

In [4]:
import pandas as pd
df = pd.DataFrame({'key': ['K0', 'K1', 'K2', 'K3', 'K4', 'K5'],
                   'A': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']})

df

,key,A
0,K0,A0
1,K1,A1
2,K2,A2
3,K3,A3
4,K4,A4
5,K5,A5


In [6]:
other = pd.DataFrame({'key': ['K0', 'K1', 'K2'],
                       'B': ['B0', 'B1', 'B2']})

other

,key,B
0,K0,B0
1,K1,B1
2,K2,B2


In [7]:
df.join(other, lsuffix='_caller', rsuffix='_other')


,key_caller,A,key_other,B
0,K0,A0,K0,B0
1,K1,A1,K1,B1
2,K2,A2,K2,B2
3,K3,A3,NaN,NaN
4,K4,A4,NaN,NaN
5,K5,A5,NaN,NaN


In [8]:
df.set_index('key').join(other.set_index('key'))

,A,B
key,,
K0,A0,B0
K1,A1,B1
K2,A2,B2
K3,A3,NaN
K4,A4,NaN
K5,A5,NaN


In [9]:
df.join(other.set_index('key'), on='key')

,key,A,B
0,K0,A0,B0
1,K1,A1,B1
2,K2,A2,B2
3,K3,A3,NaN
4,K4,A4,NaN
5,K5,A5,NaN


#### Using non-unique key values shows how they are matched.

In [10]:
df = pd.DataFrame({'key': ['K0', 'K1', 'K1', 'K3', 'K0', 'K1'],
                   'A': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']})
df

,key,A
0,K0,A0
1,K1,A1
2,K1,A2
3,K3,A3
4,K0,A4
5,K1,A5


In [11]:
df.join(other.set_index('key'), on='key', validate='m:1')

,key,A,B
0,K0,A0,B0
1,K1,A1,B1
2,K1,A2,B1
3,K3,A3,NaN
4,K0,A4,B0
5,K1,A5,B1


In [18]:
import pandas as pd

# DataFrame awal
df = pd.DataFrame({
    'key': ['K0', 'K1', 'K2', 'K3', 'K4', 'K5'],
    'A': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']
})

other = pd.DataFrame({
    'key': ['K0', 'K1', 'K2'],
    'B': ['B0', 'B1', 'B2']
})

# Join menggunakan key
joined = df.join(other.set_index('key'), on='key')

# Ubah ke matriks (numpy array)
matrix = joined.values

print("DataFrame hasil join:\n", joined, "\n")
print("Dalam bentuk matriks:\n", matrix)


DataFrame hasil join:
   key   A    B
0  K0  A0   B0
1  K1  A1   B1
2  K2  A2   B2
3  K3  A3  NaN
4  K4  A4  NaN
5  K5  A5  NaN 

Dalam bentuk matriks:
 [['K0' 'A0' 'B0']
 ['K1' 'A1' 'B1']
 ['K2' 'A2' 'B2']
 ['K3' 'A3' nan]
 ['K4' 'A4' nan]
 ['K5' 'A5' nan]]


#### Matriks

In [ ]:
import numpy as np

# Data df (key, A)
df = [
    ["K0", "A0"],
    ["K1", "A1"],
    ["K2", "A2"],
    ["K3", "A3"],
    ["K4", "A4"],
    ["K5", "A5"]
]

# Data other (key, B)
other = [
    ["K0", "B0"],
    ["K1", "B1"],
    ["K2", "B2"]
]

# Ubah other jadi dictionary untuk lookup
lookup = {k: b for k, b in other}

# Bangun matriks gabungan
matrix_list = []
for key, a in df:
    b = lookup.get(key, np.nan)  # kalau tidak ada → NaN
    matrix_list.append([key, a, b])

# Konversi ke NumPy array
matrix_np = np.array(matrix_list)

print("Matriks:\n", matrix_np)


Matriks:
 [['K0' 'A0' 'B0']
 ['K1' 'A1' 'B1']
 ['K2' 'A2' 'B2']
 ['K3' 'A3' 'nan']
 ['K4' 'A4' 'nan']
 ['K5' 'A5' 'nan']]


In [13]:
import numpy as np
import pandas as pd

# --- DataFrame contoh ---
df = pd.DataFrame({
    'key': ['K0', 'K1', 'K2', 'K3', 'K4', 'K5'],
    'A': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']
})

other = pd.DataFrame({
    'key': ['K0', 'K1', 'K2'],
    'B': ['B0', 'B1', 'B2']
})

# --- Fungsi Matrix Join ---
def matrix_join(left_array, right_array, left_indices, right_indices):
    # Buat index mapping (boolean matrix)
    index_map = left_indices[:, np.newaxis] == right_indices[np.newaxis, :]
    print("Index Map:\n", index_map, "\n")

    # Inisialisasi result array dengan NaN
    result = np.full((left_array.shape[0], 
                      left_array.shape[1] + right_array.shape[1]),
                     np.nan, dtype=object)
    print("Initial Result Array:\n", result, "\n")

    # Copy left array ke result
    result[:, :left_array.shape[1]] = left_array
    print("After Copying Left Array:\n", result, "\n")

    # Isi kolom right berdasarkan index_map
    for i in range(len(left_indices)):
        matches = np.where(index_map[i])[0]   # cari kecocokan index
        if len(matches) > 0:
            result[i, left_array.shape[1]:] = right_array[matches[0]]
        print(f"After Filling Row {i}:\n", result, "\n")

    return result

# --- Ekstrak values & indices dari DataFrame ---
left_arr = df.values        # [['K0','A0'], ..., ['K5','A5']]
right_arr = other.values    # [['K0','B0'], ['K1','B1'], ['K2','B2']]
left_idx = df.index.values  # [0,1,2,3,4,5]
right_idx = other.index.values  # [0,1,2]

# --- Panggil fungsi ---
matrix_join_result = matrix_join(left_arr, right_arr, left_idx, right_idx)
print("Final Result:\n", matrix_join_result)


Index Map:
 [[ True False False]
 [False  True False]
 [False False  True]
 [False False False]
 [False False False]
 [False False False]] 

Initial Result Array:
 [[nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]
 [nan nan nan nan]] 

After Copying Left Array:
 [['K0' 'A0' nan nan]
 ['K1' 'A1' nan nan]
 ['K2' 'A2' nan nan]
 ['K3' 'A3' nan nan]
 ['K4' 'A4' nan nan]
 ['K5' 'A5' nan nan]] 

After Filling Row 0:
 [['K0' 'A0' 'K0' 'B0']
 ['K1' 'A1' nan nan]
 ['K2' 'A2' nan nan]
 ['K3' 'A3' nan nan]
 ['K4' 'A4' nan nan]
 ['K5' 'A5' nan nan]] 

After Filling Row 1:
 [['K0' 'A0' 'K0' 'B0']
 ['K1' 'A1' 'K1' 'B1']
 ['K2' 'A2' nan nan]
 ['K3' 'A3' nan nan]
 ['K4' 'A4' nan nan]
 ['K5' 'A5' nan nan]] 

After Filling Row 2:
 [['K0' 'A0' 'K0' 'B0']
 ['K1' 'A1' 'K1' 'B1']
 ['K2' 'A2' 'K2' 'B2']
 ['K3' 'A3' nan nan]
 ['K4' 'A4' nan nan]
 ['K5' 'A5' nan nan]] 

After Filling Row 3:
 [['K0' 'A0' 'K0' 'B0']
 ['K1' 'A1' 'K1' 'B1']
 ['K2' 'A2' 'K2' 'B2']
 ['K3' 